In [1]:
!pip install -q transformers torch accelerate

import os
import pandas as pd
import torch
from transformers import pipeline
from google.colab import drive

In [8]:
# install required libraries (can take time)
%pip install pandas
%pip install numpy
%pip install transformers
%pip install torch
%pip install evaluate
%pip install scikit-learn
%pip install datasets

# install specific keras version to avoid
# clash of package versions with transformers
%pip install tf-keras==2.16.0 --no-dependencies

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.9 MB/s eta 0:00:00
  Attempting uninstall: tf-keras
    Found existing installation: tf_keras 2.19.0
    Uninstalling tf_keras-2.19.0:
      Successfully uninstalled tf_keras-2.19.0


In [9]:
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
#Mount the drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
file_path = '/content/gdrive/My Drive/AI_LLMs/AI_project/training_data_merged.csv' #file is located at data/temp/training_data_merged.csv in the repo, but we need to specify the full path in Colab
data = pd.read_csv(file_path)

# display the first few rows of the data frame
print(data.head())

        doc_id  Unnamed: 0.1  Unnamed: 0  \
0  LKA-1966.62          1172        4171   
1  SLB-1999.35          9248       25969   
2   SLV-1998.5          8860       25102   
3  VEN-1995.24          7897       23035   
4  FIN-1982.71          3907       12518   

                                                text iso_code  year  \
0  We are at present experiencing one of the most...      LKA  1966   
1  New information and communications technologie...      SLB  1999   
2  We are carrying out a successful plan for mode...      SLV  1998   
3  We have also spoken of the justification of ag...      VEN  1995   
4  We sincerely hope that the forth-coming minist...      FIN  1982   

   final_label sentiment_label  sentiment_comment  
0  neo-liberal        negative                NaN  
1  neo-liberal        negative                NaN  
2  neo-liberal        positive                NaN  
3  neo-liberal        negative                NaN  
4  neo-liberal        negative                Na

In [24]:
data = data[data["sentiment_label"] != "neutral"]

In [25]:
# load the CSV file (if you run code locally)
# file_path = "data/data_ipap_housing_2020.csv"  # Replace with the path to your CSV file
# data = pd.read_csv(file_path)

# shuffle the rows of the dataframe
shuffled_data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# sample 1000 observations to test code (do not use for final model)
# shuffled_data = shuffled_data.sample(n=1000, random_state=42)

# Split the data into training (70%), evaluation (10%), and test (20%) sets
train_size = 0.7
eval_size = 0.1

# First, split into training and remaining sets
train_data, remaining_data = train_test_split(
    shuffled_data, test_size = (1 - train_size), random_state=42
)

# Then, split the remaining data into evaluation and test sets
eval_data, test_data = train_test_split(
    remaining_data, test_size = (1 - eval_size / (1 - train_size)), random_state=42
)

# Output the sizes of the resulting sets
len(train_data), len(eval_data), len(test_data)

(159, 22, 47)

In [26]:
train = train_data[['text', 'sentiment_label']]
eval = eval_data[['text', 'sentiment_label']]
test = test_data[['text', 'sentiment_label']]

In [27]:
le = LabelEncoder()

# fit on training labels only, then transform all splits
train.loc[:, 'labels'] = le.fit_transform(train['sentiment_label'])
eval.loc[:, 'labels']  = le.transform(eval['sentiment_label'])
test.loc[:, 'labels']  = le.transform(test['sentiment_label'])

# reconstruct dicts for model config (e.g. id2label in AutoConfig)
id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in id2label.items()}

/tmp/ipykernel_457/2639033293.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.loc[:, 'labels'] = le.fit_transform(train['sentiment_label'])
/tmp/ipykernel_457/2639033293.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval.loc[:, 'labels']  = le.transform(eval['sentiment_label'])
/tmp/ipykernel_457/2639033293.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documenta

In [28]:
id2label

{0: 'negative', 1: 'positive'}

In [29]:
label2id

{'negative': 0, 'positive': 1}

In [30]:
# get unique number of labels in train data
n_classes = train['labels'].nunique()

print(n_classes)

2


In [31]:

# Fail immediately if no GPU is available
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. In Colab go to Runtime > Change runtime type > GPU, "
        "then reconnect and run again."
    )

device = 0  # first CUDA GPU
print("Using GPU:", torch.cuda.get_device_name(0))

Using GPU: Tesla T4


In [67]:
# Import Model
#Pick an explicit model for reproducibility
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

In [68]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = n_classes,
    id2label = id2label,
    label2id = label2id)

print(model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [69]:
# convert to Hugging Face Dataset format for fine-tuning
train_convert = Dataset.from_pandas(train)
eval_convert = Dataset.from_pandas(eval)
test_convert = Dataset.from_pandas(test)

In [70]:
# set batch size
batch_size = 8

# get logging steps
logging_steps = len(train_convert) // batch_size

# specify name for fine-tuned model
model_name_finetuned = f"{model_name}-finetuned"

# specify training arguments - these are close to the defaults
training_args = TrainingArguments(
    output_dir=model_name_finetuned,              # Directory to save model checkpoints and logs.
    num_train_epochs=2,                           # Number of full training passes over the dataset.
    learning_rate=5e-5,                           # Initial learning rate.
    per_device_train_batch_size=batch_size,       # Batch size for training per device (GPU/CPU).
    per_device_eval_batch_size=batch_size,        # Batch size for evaluation per device.
    eval_strategy="epoch",                        # Run evaluation at the end of each epoch.
    logging_steps=logging_steps,                  # Log training metrics every `logging_steps` steps.
    push_to_hub=False, # Disable automatic pushing of model to the Hugging Face Hub.
)

In [71]:
def compute_metrics(eval_pred):
    f1 = evaluate.load("f1")
    accuracy = evaluate.load("accuracy")

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1val = f1.compute(references=labels, predictions=predictions, average = "weighted")
    accval = accuracy.compute(references=labels, predictions=predictions)
    return {"F1": f1val, "accuracy": accval}

In [72]:
# define tokenizer
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=True)

# tokenize the datasets
train_tokenized = train_convert.map(tokenize, batched=True)
eval_tokenized = eval_convert.map(tokenize, batched=True)
test_tokenized = test_convert.map(tokenize, batched=True)

Map:   0%|          | 0/159 [00:00<?, ? examples/s]

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

In [73]:
# specify trainer arguments and evaluation metrics
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_tokenized, # specify training data
    eval_dataset=eval_tokenized, # specify evaluation data
    processing_class=tokenizer
)

In [74]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.490652,0.269907,{'f1': 0.9550406020994257},{'accuracy': 0.9545454545454546}
2,0.093199,0.332934,{'f1': 0.9090909090909091},{'accuracy': 0.9090909090909091}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=40, training_loss=0.2832521177828312, metrics={'train_runtime': 18.1366, 'train_samples_per_second': 17.534, 'train_steps_per_second': 2.205, 'total_flos': 28384762317480.0, 'train_loss': 0.2832521177828312, 'epoch': 2.0})

In [ ]:
# save model in designated folder if you are running this locally
model_name_finetuned = f"{model_name}-finetuned-UN"

print(model_name_finetuned)

# we will save the model in Google Drive
# Define the path in Google Drive where you want to save the model
save_path = f'/content/gdrive/My Drive/AI_LLMs/AI_project/{model_name_finetuned}' #finetuned model parameters are saved in the folder of the same name in "google drive links in README.md"

# Save the model and tokenizer to the specified path
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)  # add this

distilbert-base-uncased-finetuned-sst-2-english-finetuned-UN


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/gdrive/My Drive/AI_LLMs/AI_project/distilbert-base-uncased-finetuned-sst-2-english-finetuned-UN/tokenizer_config.json',
 '/content/gdrive/My Drive/AI_LLMs/AI_project/distilbert-base-uncased-finetuned-sst-2-english-finetuned-UN/tokenizer.json')

In [76]:
predictions = trainer.predict(test_tokenized)
print(predictions.predictions.shape, predictions.label_ids.shape)

preds = np.argmax(predictions.predictions, axis=-1)

print(preds)

(47, 2) (47,)
[0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 1 0 1]


In [77]:
preds_labels = [id2label[p] for p in preds]

print(preds_labels)

['negative', 'negative', 'negative', 'negative', 'negative', 'positive', 'negative', 'negative', 'negative', 'negative', 'positive', 'positive', 'negative', 'negative', 'negative', 'negative', 'negative', 'positive', 'negative', 'negative', 'negative', 'positive', 'negative', 'negative', 'negative', 'negative', 'negative', 'positive', 'negative', 'negative', 'negative', 'negative', 'positive', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'positive', 'negative', 'positive']


In [78]:
# generate the classification report for test set
report = classification_report(
    y_true=test_tokenized['labels'],
    y_pred=preds,
    target_names=[id2label[i] for i in sorted(id2label)]
)

print(report)

              precision    recall  f1-score   support

    negative       0.82      0.97      0.89        32
    positive       0.89      0.53      0.67        15

    accuracy                           0.83        47
   macro avg       0.85      0.75      0.78        47
weighted avg       0.84      0.83      0.82        47



In [80]:
# create cross-table/confusion matrix
confusion_table = pd.crosstab(
    test_tokenized['sentiment_label'],
    preds_labels,
    rownames=['Actual'],
    colnames=['Predicted'],
    dropna=False
)

print(confusion_table)

Predicted  negative  positive
Actual                       
negative         31         1
positive          7         8


In [87]:
"/content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences.csv"

'/content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences.csv'

In [88]:
df = pd.read_csv("/content/gdrive/MyDrive/AI_LLMs/AI_project/neo_geo_class_sentences.csv")

In [93]:
model_path = "/content/gdrive/My Drive/AI_LLMs/AI_project/distilbert-base-uncased-finetuned-sst-2-english-finetuned-UN"
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [91]:
model_sentiment = AutoModelForSequenceClassification.from_pretrained(model_path)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [94]:
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model_sentiment,
    tokenizer=tokenizer,
    device=device
)

In [97]:
def classify_sentences(df, column):
    results = sentiment_classifier(df[column].tolist())

    # work on a copy to avoid mutating the original
    df = df.copy()
    df['label_sentiment'] = [result['label'] for result in results]
    df['score'] = [result['score'] for result in results]
    return df

In [98]:
df.shape

(21200, 9)

In [105]:
labeled_df = classify_sentences(df, column = "text")

In [107]:
len(labeled_df[labeled_df['label_sentiment'] == 'negative'])

11606

In [ ]:
labeled_df.to_csv("/content/gdrive/My Drive/AI_LLMs/AI_project/data_classfied_finetuned.csv", index=False) # the same file is saved in the repo at data/temp/data_classfied_finetuned.csv, but we need to specify the full path in Colab

In [ ]:
# Your three input files
input_files = [

    "/content/gdrive/MyDrive/Colab Notebooks/neo_geo_classfied_main.csv",
    "/content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_chunks.csv",
    "/content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences.csv",
]


In [ ]:
# Columns to keep if they exist
preferred_columns = ["doc_id", "text", "iso_code", "year", "final_label"]

print("Files to process:")
for f in input_files:
    print("-", f, "| exists:", os.path.exists(f))

saved_files = []  # Initialize saved_files here to ensure it's empty before the loop

for file_path in input_files:
    if not os.path.exists(file_path):
        print(f"SKIPPED - file not found: {file_path}")
        continue

    print(f"\nProcessing: {file_path}")
    df = pd.read_csv(file_path)

    if "text" not in df.columns:
        print(f"SKIPPED - no 'text' column: {file_path}")
        continue

    # Clean text column
    texts = df["text"].fillna("").astype(str).tolist()

    # Run sentiment analysis
    results = sentiment_classifier(
        texts,
        batch_size=32,
        truncation=True
    )

    # Keep original columns that actually exist
    cols_to_keep = [col for col in preferred_columns if col in df.columns]
    results_df = df[cols_to_keep].copy()

    # Add predictions
    results_df["sentiment_label"] = [r["label"] for r in results]
    results_df["confidence_score"] = [r["score"] for r in results]

    # Save output beside original file
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    output_path = f"/content/gdrive/MyDrive/Colab Notebooks/{base_name}_sentiment_results.csv"

    results_df.to_csv(output_path, index=False)
    saved_files.append(output_path)

    # Quick summary for the current file being processed
    print(f"Saved: {output_path}")
    print(results_df["sentiment_label"].value_counts(dropna=False))
    display(results_df.head())

Files to process:
- /content/gdrive/MyDrive/Colab Notebooks/neo_geo_classfied_main.csv | exists: True
- /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_chunks.csv | exists: True
- /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences.csv | exists: True

Processing: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_classfied_main.csv
Saved: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_classfied_main_sentiment_results.csv
sentiment_label
POSITIVE    8020
NEGATIVE    5253
Name: count, dtype: int64


,doc_id,text,iso_code,year,final_label,sentiment_label,confidence_score
0,CHL-1946.14,Our foreign exchange reserves are not always p...,CHL,1946,neo-liberal,POSITIVE,0.991643
1,EGY-1946.18,Egypt is confident that the General Assembly a...,EGY,1946,geo-economic,POSITIVE,0.993147
2,GBR-1946.27,That will save us money — perhaps a lot; more ...,GBR,1946,neo-liberal,NEGATIVE,0.525938
3,GBR-1946.28,The first is to declare again the basic princi...,GBR,1946,neo-liberal,POSITIVE,0.995486
4,PHL-1946.12,We trust that this General Assembly and the va...,PHL,1946,neo-liberal,NEGATIVE,0.935999



Processing: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_chunks.csv
Saved: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_chunks_sentiment_results.csv
sentiment_label
POSITIVE    7672
NEGATIVE    5077
Name: count, dtype: int64


,doc_id,text,iso_code,year,final_label,sentiment_label,confidence_score
0,CHL-1946.14,We do not sell our exportable products at suff...,CHL,1946,neo-liberal,NEGATIVE,0.999561
1,EGY-1946.18,When we consider some of the economic problems...,EGY,1946,geo-economic,NEGATIVE,0.969219
2,GBR-1946.27,The first is to declare again the basic princi...,GBR,1946,neo-liberal,POSITIVE,0.990102
3,GBR-1946.28,The first is to declare again the basic princi...,GBR,1946,neo-liberal,POSITIVE,0.996226
4,PHL-1946.12,This is not the sort of economic interdependen...,PHL,1946,neo-liberal,NEGATIVE,0.991665



Processing: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences.csv
Saved: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences_sentiment_results.csv
sentiment_label
POSITIVE    12708
NEGATIVE     8492
Name: count, dtype: int64


,doc_id,text,iso_code,year,final_label,sentiment_label,confidence_score
0,CHL-1946.14,We do not sell our exportable products at suff...,CHL,1946,neo-liberal,NEGATIVE,0.999561
1,EGY-1946.18,When we consider some of the economic problems...,EGY,1946,geo-economic,NEGATIVE,0.969219
2,GBR-1946.27,The first is to declare again the basic princi...,GBR,1946,neo-liberal,POSITIVE,0.990102
3,GBR-1946.28,"Ever since the Atlantic Charter, every Member ...",GBR,1946,neo-liberal,POSITIVE,0.998854
4,PHL-1946.12,This is not the sort of economic interdependen...,PHL,1946,neo-liberal,NEGATIVE,0.999510
